# Create the FD001 engine-disjoint protocol and reference metadata.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd

AUDIT_DIR = Path("revision_outputs") / "01_FD001_data_audit"
OUTPUT_DIR = Path("revision_outputs") / "02_FD001_protocol"
DATASET = "FD001"
PROTOCOL_VERSION = "FD001_reference_splits_v1"
SPLIT_SEED = 42
N_OUTER_FOLDS = 10
N_EARLY_STOP = 10
N_SELECTION = 10
PRIMARY_RUL_THRESHOLD = 30
RUL_THRESHOLDS = (20, 30, 40)
EXPECTED_ROWS = 20631
EXPECTED_UNITS = 100

summary_path = AUDIT_DIR / "validation_summary.json"
metadata_path = AUDIT_DIR / "train_evaluation_metadata.csv"
print("Önceki aşama:", AUDIT_DIR.resolve())


In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def validate_train_metadata(metadata, expected_rows, expected_units):
    required = ["dataset", "split", "source_row", "UnitNumber", "Time",
                "engine_key", "sample_key", "RUL"]
    require(set(required).issubset(metadata.columns), "Eğitim metadata sütunları eksik.")
    require(len(metadata) == expected_rows, "Eğitim metadata satır sayısı beklenenle uyuşmuyor.")
    require(metadata[required].notna().all().all(), "Eğitim metadata içinde eksik değer var.")
    require(metadata["dataset"].eq(DATASET).all() and metadata["split"].eq("train").all(),
            "Bu notebook yalnız FD001 eğitim metadatasını kullanır.")
    for column in ["source_row", "UnitNumber", "Time", "RUL"]:
        values = pd.to_numeric(metadata[column], errors="raise").to_numpy(dtype=float)
        require(np.isfinite(values).all() and np.all(values == np.floor(values)),
                f"{column} sonlu tam sayılardan oluşmalı.")
        metadata[column] = values.astype("int64")
    require(np.array_equal(metadata["source_row"].to_numpy(), np.arange(expected_rows)),
            "source_row sırası değişmiş. 01. aşamayı tekrar çalıştırın.")
    require(np.array_equal(np.sort(metadata["UnitNumber"].unique()),
                           np.arange(1, expected_units + 1)), "Motor kimlikleri beklenenle uyuşmuyor.")
    require(not metadata.duplicated(["UnitNumber", "Time"]).any(), "Tekrarlanan motor/çevrim var.")
    expected_engines = DATASET + ":train:" + metadata["UnitNumber"].astype(str)
    expected_samples = expected_engines + ":" + metadata["Time"].astype(str)
    require(metadata["engine_key"].eq(expected_engines).all(), "Motor anahtarları geçersiz.")
    require(metadata["sample_key"].eq(expected_samples).all() and metadata["sample_key"].is_unique,
            "Örnek anahtarları geçersiz veya tekrarlanmış.")
    for engine, group in metadata.groupby("engine_key", sort=False):
        require(np.array_equal(group["Time"].to_numpy(), np.arange(1, len(group) + 1)),
                f"{engine}: çevrim sırası bozuk.")
        require(np.array_equal(group["RUL"].to_numpy(),
                               group["Time"].max() - group["Time"].to_numpy()),
                f"{engine}: eğitim RUL değeri son çevrim eksi mevcut çevrim olmalı.")
    return metadata


def stable_order(keys, salt, seed=SPLIT_SEED):

    def ordering_key(engine):
        value = f"{seed}|{salt}|{engine}".encode("utf-8")
        return (hashlib.sha256(value).hexdigest(), engine)
    return sorted(keys, key=ordering_key)


def make_engine_splits(keys, n_folds=N_OUTER_FOLDS,
                       n_early=N_EARLY_STOP, n_selection=N_SELECTION):
    keys = sorted(set(keys))
    require(len(keys) >= n_folds, "Motor sayısı dış fold sayısından az.")
    outer_groups = np.array_split(np.asarray(stable_order(keys, "outer"), dtype=object), n_folds)
    rows = []
    for fold, validation in enumerate(outer_groups, start=1):
        validation = set(validation.tolist())
        inner = stable_order(set(keys) - validation, f"inner:{fold}")
        require(len(inner) > n_early + n_selection, "fit motoru kalmıyor.")
        role_sets = {
            "early_stop": set(inner[:n_early]),
            "selection": set(inner[n_early:n_early + n_selection]),
            "fit": set(inner[n_early + n_selection:]),
            "outer_valid": validation,
        }
        for role, engines in role_sets.items():
            rows.extend({"scheme": "cv", "fold": fold, "engine_key": engine, "role": role}
                        for engine in sorted(engines))
    final = stable_order(keys, "final")
    role_sets = {
        "early_stop": final[:n_early],
        "selection": final[n_early:n_early + n_selection],
        "fit": final[n_early + n_selection:],
    }
    for role, engines in role_sets.items():
        rows.extend({"scheme": "final", "fold": 0, "engine_key": engine, "role": role}
                    for engine in engines)
    return pd.DataFrame(rows).sort_values(["scheme", "fold", "engine_key"]).reset_index(drop=True)


def validate_engine_splits(splits, engine_keys, n_folds, n_early, n_selection):
    keys = set(engine_keys)
    require(not splits.duplicated(["scheme", "fold", "engine_key"]).any(),
            "Aynı motor bir deneyde birden fazla role atanmış.")
    require(set(splits["engine_key"]) == keys, "Atanan motor kimlikleri farklı.")
    require(set(splits["scheme"]) == {"cv", "final"}, "Bölme düzeni adları geçersiz.")
    cv = splits[splits["scheme"].eq("cv")]
    require(set(cv["fold"]) == set(range(1, n_folds + 1)), "Dış fold sayısı/numarası hatalı.")
    for fold, part in cv.groupby("fold"):
        require(len(part) == len(keys) and set(part["engine_key"]) == keys,
                f"Fold {fold}: tüm motorlar tam bir kez atanmalı.")
        counts = part["role"].value_counts().to_dict()
        require(set(counts) == {"fit", "early_stop", "selection", "outer_valid"},
                f"Fold {fold}: roller eksik veya geçersiz.")
        require(counts["early_stop"] == n_early and counts["selection"] == n_selection,
                f"Fold {fold}: iç küme büyüklükleri yanlış.")
        require(counts["fit"] > 0 and counts["outer_valid"] > 0, "Boş fit/değerlendirme kümesi.")
    outer_counts = cv[cv["role"].eq("outer_valid")]["engine_key"].value_counts()
    require(set(outer_counts.index) == keys and outer_counts.eq(1).all(),
            "Her motor tam bir kez dış değerlendirmede yer almalı.")
    final = splits[splits["scheme"].eq("final")]
    counts = final["role"].value_counts().to_dict()
    require(len(final) == len(keys) and set(final["engine_key"]) == keys
            and final["fold"].eq(0).all(), "Nihai eğitim motorları hatalı.")
    require(counts == {"fit": len(keys)-n_early-n_selection,
                       "early_stop": n_early, "selection": n_selection},
            "Nihai deney rol büyüklükleri hatalı.")


def write_once_or_equal(path, payload):
    if path.exists():
        require(path.read_bytes() == payload,
                f"{path}: mevcut kayıt farklı. Üzerine yazılmadı. Değişiklik gerekçesiyle yeni sürüm/klasör kullanın.")
    else:
        path.write_bytes(payload)


In [ ]:
require((DATASET, SPLIT_SEED, N_OUTER_FOLDS, N_EARLY_STOP, N_SELECTION,
         PRIMARY_RUL_THRESHOLD, RUL_THRESHOLDS, EXPECTED_ROWS, EXPECTED_UNITS)
        == ("FD001", 42, 10, 10, 10, 30, (20, 30, 40), 20631, 100),
        "Bu sürümün deney ayarları değişmiş. Yalnız yolları değiştirin; protokol değişikliği yeni sürüm gerektirir.")
for path in [summary_path, metadata_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Eksik dosya: {path}. AUDIT_DIR yolunu ve 01. aşama çıktılarını kontrol edin.")

audit = json.loads(summary_path.read_text(encoding="utf-8"))
require(audit.get("stage") == "01_data_audit" and audit.get("dataset") == DATASET
        and audit.get("status") == "PASS", "Önce FD001 veri doğrulaması PASS olmalı.")
require(audit["train"]["shape"] == [EXPECTED_ROWS, 26]
        and audit["train"]["units"] == EXPECTED_UNITS, "Önceki özet boyutları beklenenden farklı.")
require(audit.get("rul_is_separate_from_sensor_inputs") is True
        and audit.get("all_21_sensors_retained") is True, "01. aşamanın girdi ayrımı korunmamış.")
require(audit.get("models_trained") is False and audit.get("binary_anomaly_labels_created") is False,
        "Beklenen başlangıç özeti değişmiş.")
source_hashes = audit.get("file_sha256", {})
require(set(source_hashes) == {"train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"},
        "Kaynak dosya hash kayıtları eksik.")

train_metadata = validate_train_metadata(pd.read_csv(metadata_path), EXPECTED_ROWS, EXPECTED_UNITS)

reference = train_metadata.copy()
for threshold in RUL_THRESHOLDS:
    reference[f"late_life_le_{threshold}"] = reference["RUL"].le(threshold).astype("int8")
    require(reference[f"late_life_le_{threshold}"].nunique() == 2,
            f"{threshold} çevrim referansında iki sınıf da bulunmalı.")

engine_keys = sorted(reference["engine_key"].unique().tolist())
engine_splits = make_engine_splits(engine_keys)
validate_engine_splits(engine_splits, engine_keys, N_OUTER_FOLDS, N_EARLY_STOP, N_SELECTION)
require(engine_splits["engine_key"].str.startswith(DATASET + ":train:").all(),
        "Resmî test motoru atamalara karışmış.")
print("Motor bölmeleri ve AE'den bağımsız RUL vekil referansları oluşturuldu.")


In [ ]:
role_rows = []
for (scheme, fold, role), assigned in engine_splits.groupby(["scheme", "fold", "role"], sort=True):
    subset = reference[reference["engine_key"].isin(assigned["engine_key"])]
    record = {"scheme": scheme, "fold": int(fold), "role": role,
              "engines": int(assigned["engine_key"].nunique()), "rows": int(len(subset))}
    for threshold in RUL_THRESHOLDS:
        col = f"late_life_le_{threshold}"
        require(subset[col].nunique() == 2, f"{scheme}/{fold}/{role}: {col} tek sınıf içeriyor.")
        record[f"positive_le_{threshold}"] = int(subset[col].sum())
        record[f"prevalence_le_{threshold}"] = float(subset[col].mean())
    role_rows.append(record)
role_summary = pd.DataFrame(role_rows)
primary_col = f"late_life_le_{PRIMARY_RUL_THRESHOLD}"

reference_counts = {
    str(threshold): {"positive": int(reference[f"late_life_le_{threshold}"].sum()),
                     "negative": int((reference[f"late_life_le_{threshold}"] == 0).sum()),
                     "prevalence": float(reference[f"late_life_le_{threshold}"].mean())}
    for threshold in RUL_THRESHOLDS
}
print("Eğitim referansları (anomali ground truth değildir):")
print(json.dumps(reference_counts, ensure_ascii=False, indent=2))
print("\nMotor sayıları:")
print(role_summary.pivot(index=["scheme", "fold"], columns="role", values="engines").fillna(0).astype(int).to_string())


In [ ]:
protocol = {
    "version": PROTOCOL_VERSION,
    "scope": "reference_and_engine_splits; no model training",
    "dataset": DATASET,
    "split_seed": SPLIT_SEED,
    "split_method": "SHA256(seed|salt|engine_key) ordering; 10 outer groups; no label stratification",
    "outer_folds": N_OUTER_FOLDS,
    "cv_roles_engines": {"fit": 70, "early_stop": 10, "selection": 10, "outer_valid": 10},
    "final_roles_engines": {"fit": 80, "early_stop": 10, "selection": 10},
    "final_refit_after_selection": False,
    "feature_columns": [f"Sensor{i}" for i in range(1, 22)],
    "excluded_from_model_inputs": ["UnitNumber", "Time", "OpSet1", "OpSet2", "OpSet3", "RUL",
                                    "engine_key", "sample_key", "source_row", "reference_labels"],
    "constant_sensor_removal": False,
    "fit_scope": {"input_scaler": "fit", "PCA": "fit", "AE_weights": "fit",
                  "latent_scaler": "fit", "detectors": "fit"},
    "early_stopping": "reconstruction loss on early_stop engines; explicit validation_data",
    "reference": {"name": "RUL-derived late-life proxy", "primary_threshold_cycles": PRIMARY_RUL_THRESHOLD,
                  "threshold_operator": "<=", "sensitivity_thresholds_cycles": [20, 40],
                  "includes_RUL_zero": True, "all_cycles_retained": True,
                  "known_anomaly_onset": False, "negative_class_means_verified_healthy": False,
                  "same_reference_for_all_models": True, "reference_independent_of_model_outputs": True,
                  "test_formula": "terminal_official_RUL + last_observed_cycle - current_cycle"},
    "label_use": {"fitting": "none", "early_stopping": "none",
                  "hyperparameter_and_threshold_selection": "RUL proxy labels on selection engines only",
                  "reporting": "outer_valid or official test; never reused for within-fold selection",
                  "description": "unsupervised fitting with label-informed model/threshold selection"},
    "primary_metric": "pooled sample-level average_precision_score (AP), positive=late-life proxy",
    "secondary_metrics": ["ROC-AUC", "positive Precision", "positive Recall", "positive F1", "Macro-F1"],
    "hpo_objective": "selection AP under primary reference; ties use predeclared candidate order",
    "threshold_objective": "selection positive-class F1 under primary reference; ties choose higher threshold",
    "decision_rule": "positive if anomaly-oriented continuous score >= threshold",
    "sensitivity_model_and_threshold_reselection": False,
    "test_access": "not read in this stage; excluded from revised model/threshold selection",
    "prior_official_test_exposure": "test results already examined in original study; not a pristine new holdout",
    "predeclared_baselines": ["standardized raw sensors", "PCA representation", "AE representation",
                              "AE reconstruction MSE scored against independent RUL proxy"],
    "pilot_training_seed": 42,
    "full_experiment_training_seeds": [42, 43, 44],
    "fold_uncertainty": "fold SD descriptive; folds are not independent replicates for automatic Wilcoxon claims",
    "pending_before_training": ["freeze HPO candidate spaces and selection logs",
                                "define consistent continuous scores and DBSCAN protocol",
                                "implement fit/early_stop/selection/outer_valid data-use checks"],
    "source_raw_file_sha256_from_stage01": source_hashes,
    "stage01_summary_sha256": sha256_file(summary_path),
    "train_metadata_sha256": sha256_file(metadata_path),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
payloads = {
    "experiment_protocol.json": (json.dumps(protocol, ensure_ascii=False, indent=2) + "\n").encode("utf-8"),
    "engine_splits.csv": engine_splits.to_csv(index=False, lineterminator="\n").encode("utf-8"),
    "train_evaluation_reference.csv": reference.to_csv(index=False, lineterminator="\n").encode("utf-8"),
    "role_summary.csv": role_summary.to_csv(index=False, lineterminator="\n").encode("utf-8"),
}

for filename, payload in payloads.items():
    path = OUTPUT_DIR / filename
    require(not path.exists() or path.read_bytes() == payload,
            f"{path}: mevcut kayıt farklı; yeni protokol sürümü/klasörü kullanın.")
for filename, payload in payloads.items():
    write_once_or_equal(OUTPUT_DIR / filename, payload)

protocol_digest = hashlib.sha256(canonical_json(protocol).encode("utf-8")).hexdigest()
summary = {
    "stage": "02_reference_and_engine_splits",
    "status": "PASS",
    "dataset": DATASET,
    "protocol_version": PROTOCOL_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
    "train_rows": len(reference), "train_engines": len(engine_keys),
    "outer_folds": N_OUTER_FOLDS,
    "cv_engines_per_role": protocol["cv_roles_engines"],
    "final_engines_per_role": protocol["final_roles_engines"],
    "each_engine_outer_valid_once": True,
    "engine_roles_disjoint_within_experiment": True,
    "reference_counts": reference_counts,
    "primary_reference": primary_col,
    "primary_metric": "AP",
    "true_anomaly_onset_labels": False,
    "model_selection_uses_reference_labels": True,
    "test_files_read_in_this_stage": False,
    "models_trained": False,
    "canonical_protocol_sha256": protocol_digest,
    "output_file_sha256": {name: hashlib.sha256(payload).hexdigest() for name, payload in payloads.items()},
    "next_stage": "Fold içinde öğrenen modeller, HPO adayları ve skor/eşik kuralları; önce FD001 pilotu",
}
(OUTPUT_DIR / "protocol_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("PROTOCOL AUDIT:", summary["status"])
print("Ana referans:", primary_col, "| Model eğitilmedi | Resmî test okunmadı")
print("Kontrol için paylaşılacak dosya:", (OUTPUT_DIR / "protocol_summary.json").resolve())
